In [7]:
"""
Reproducible stratified sampling (Option A: PROPORTIONAL allocation)
--------------------------------------------------------------------
- Builds 8 strata: Y{0/1}_B{0/1}_A{0/1}
- Allocates sample counts proportional to stratum size so total == TARGET_TOTAL
- Draws a reproducible random sample per stratum (fixed seed)
- ALWAYS overwrites outputs
- In the sample file, columns are ordered: html_url, full_name, stratum, YAML_pred, Build_pred, AT_pred

Dataset columns required:
- instru_t_ci_signal       -> CI YAML signal (bool/0/1)
- instru_t_signal_config   -> Build/Gradle instrumentation signal (bool/0/1)
- Intru_test               -> androidTest present (bool/0/1)
Optional identifiers:
- full_name, html_url
"""

import os
import hashlib
import pandas as pd
import numpy as np

# ---------- CONFIG ----------
DATA_PATH  = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Target overall sample size for 90% CI, ~±5 pp MOE (SRS worst-case)
TARGET_TOTAL = 255

# Reproducible sampling
BASE_SEED = 20250820

# ---------- LOAD & PREP ----------
df = pd.read_csv(DATA_PATH)

# Helper: normalize to 0/1
def to_bin(s):
    if s.dtype == bool:
        return s.astype(int)
    if s.dtype.kind in "biufc":
        return (s.fillna(0) != 0).astype(int)
    return s.astype(str).str.strip().str.lower().isin(["1","true","t","yes","y"]).astype(int)

req_cols = {
    "YAML_pred":  "instru_t_ci_signal",
    "Build_pred": "instru_t_signal_config",
    "AT_pred":    "Intru_test",
}
missing = [v for v in req_cols.values() if v not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["YAML_pred"]  = to_bin(df[req_cols["YAML_pred"]])
df["Build_pred"] = to_bin(df[req_cols["Build_pred"]])
df["AT_pred"]    = to_bin(df[req_cols["AT_pred"]])

# Ensure both html_url and full_name exist (blank if absent)
if "html_url" not in df.columns:
    df["html_url"] = ""
if "full_name" not in df.columns:
    df["full_name"] = ""

# Stratum label
df["stratum"] = df.apply(lambda r: f"Y{int(r['YAML_pred'])}_B{int(r['Build_pred'])}_A{int(r['AT_pred'])}", axis=1)

# ---------- COHORT TABLE ----------
total_n = len(df)
cohort = (
    df.groupby(["YAML_pred","Build_pred","AT_pred","stratum"])
      .size()
      .reset_index(name="N")
      .sort_values(["YAML_pred","Build_pred","AT_pred"])
      .reset_index(drop=True)
)
cohort["pct"] = (cohort["N"] / total_n * 100).round(2)

# Overwrite cohort table
cohort_path = os.path.join(OUTPUT_DIR, "cohort_table.csv")
cohort.to_csv(cohort_path, index=False, encoding="utf-8")

# ---------- PROPORTIONAL ALLOCATION ----------
cohort["quota_exact"] = cohort["N"] * (TARGET_TOTAL / total_n)
cohort["alloc_floor"] = np.floor(cohort["quota_exact"]).astype(int)

# Largest Remainder (Hamilton) to hit TARGET_TOTAL exactly
current_sum = int(cohort["alloc_floor"].sum())
deficit = TARGET_TOTAL - current_sum

cohort["remainder"] = cohort["quota_exact"] - cohort["alloc_floor"]
cohort = cohort.sort_values(["remainder", "N"], ascending=[False, False]).reset_index(drop=True)

cohort["alloc_prop"] = cohort["alloc_floor"]
i = 0
while deficit > 0 and i < len(cohort):
    if cohort.loc[i, "alloc_prop"] < cohort.loc[i, "N"]:
        cohort.loc[i, "alloc_prop"] += 1
        deficit -= 1
    i += 1

# Restore natural order and cap by N
cohort = cohort.sort_values(["YAML_pred","Build_pred","AT_pred"]).reset_index(drop=True)
cohort["alloc_prop"] = cohort[["alloc_prop","N"]].min(axis=1).astype(int)

# Overwrite plan file
plan_path = os.path.join(OUTPUT_DIR, "proportional_plan.csv")
cohort[["stratum","N","pct","quota_exact","alloc_prop"]].to_csv(plan_path, index=False, encoding="utf-8")

# ---------- DRAW SAMPLE (REPRODUCIBLE) ----------
def stable_seed(s: str, base_seed: int = BASE_SEED) -> int:
    h = hashlib.md5(s.encode("utf-8")).hexdigest()
    return (int(h[:8], 16) ^ base_seed) & 0x7FFFFFFF

samples = []
for _, row in cohort.iterrows():
    s = row["stratum"]
    k = int(row["alloc_prop"])
    if k <= 0:
        continue
    pool = df[df["stratum"] == s][[
        "html_url", "full_name", "stratum", "YAML_pred", "Build_pred", "AT_pred"
    ]].copy()
    if len(pool) == 0:
        continue
    take = min(k, len(pool))
    rs = np.random.RandomState(stable_seed(s))
    sampled = pool.sample(n=take, replace=False, random_state=rs)
    samples.append(sampled)

if samples:
    sampled_df = pd.concat(samples, ignore_index=True)
else:
    sampled_df = pd.DataFrame(columns=["html_url","full_name","stratum","YAML_pred","Build_pred","AT_pred"])

# Ensure desired column order and overwrite sample file
cols_out = ["html_url", "full_name", "stratum", "YAML_pred", "Build_pred", "AT_pred"]
sample_out = os.path.join(OUTPUT_DIR, "stratified_sample_proportional.csv")
sampled_df.to_csv(sample_out, columns=cols_out, index=False, encoding="utf-8")

# ---------- SUMMARY ----------
print(f"Total repos: {total_n}")
print("Proportional allocations by stratum (sum should equal TARGET_TOTAL):")
print(cohort[["stratum","N","alloc_prop"]])
print(f"\nTARGET_TOTAL: {TARGET_TOTAL}")
print(f"Actual total allocated: {int(cohort['alloc_prop'].sum())}")
print(f"Saved (overwritten if existed):\n  {cohort_path}\n  {plan_path}\n  {sample_out}")


Total repos: 4518
Proportional allocations by stratum (sum should equal TARGET_TOTAL):
    stratum     N  alloc_prop
0  Y0_B0_A0  1936         109
1  Y0_B0_A1   217          12
2  Y0_B1_A0   722          41
3  Y0_B1_A1  1169          66
4  Y1_B0_A0    44           3
5  Y1_B0_A1    41           2
6  Y1_B1_A0    46           3
7  Y1_B1_A1   343          19

TARGET_TOTAL: 255
Actual total allocated: 255
Saved (overwritten if existed):
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\cohort_table.csv
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\proportional_plan.csv
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\stratified_sample_proportional.csv
